In [1]:
# Cell 1: Setup & Constants
# Notebook 02: Dim_FinancialGeography — Gold_SalesOps_Dim_FinancialGeography
# Source: FinancialGeographyHierarchy (Silver), keyed by GlobalFinancialGeographyId

from pyspark.sql import functions as F

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")

StatementMeta(, bff923fa-ee9b-41f5-9310-92a1660ca879, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo


In [2]:
# Cell 2: Load FinancialGeographyHierarchy (raw)

df_raw = (
    spark.read.format("delta").load(f"{SILVER_BASE}/FinancialGeographyHierarchy")
    .filter(F.col("IsDeleted") == False)
)

total_rows = df_raw.count()
null_gfgid = df_raw.filter(F.col("GlobalFinancialGeographyId").isNull()).count()
distinct_gfgid = df_raw.select("GlobalFinancialGeographyId").distinct().count()

print(f"Total rows (IsDeleted=False):          {total_rows:,}")
print(f"Distinct GlobalFinancialGeographyId:   {distinct_gfgid:,}")
print(f"NULL GlobalFinancialGeographyId:       {null_gfgid:,} ({null_gfgid/total_rows*100:.2f}%)")
print(f"Is GlobalFinancialGeographyId unique?  {'YES' if distinct_gfgid + null_gfgid == total_rows else 'NO — DUPLICATES EXIST'}")

display(df_raw.limit(10))

StatementMeta(, bff923fa-ee9b-41f5-9310-92a1660ca879, 4, Finished, Available, Finished, False)

Total rows (IsDeleted=False):          1,082
Distinct GlobalFinancialGeographyId:   1,082
NULL GlobalFinancialGeographyId:       0 (0.00%)
Is GlobalFinancialGeographyId unique?  YES


SynapseWidget(Synapse.DataFrame, e408885c-c5cf-45ad-8122-ba2ef7963ff3)

In [3]:
# Cell 3: Select Columns → Dim_FinancialGeography

dim_fin_geo = df_raw.select(
    F.col("GlobalFinancialGeographyId"),
    F.col("Region"),
    F.col("SubRegion"),
    F.col("CountryGroup"),
    F.col("Cluster"),
    F.col("Country"),
    F.col("CountryCode"),
)

row_count = dim_fin_geo.count()
print(f"Dim_FinancialGeography rows: {row_count:,}\n")

# NULL analysis for every column
print("Column NULL counts:")
print("-" * 55)
for col_name in dim_fin_geo.columns:
    null_count = dim_fin_geo.filter(F.col(col_name).isNull()).count()
    pct = null_count / row_count * 100 if row_count > 0 else 0
    print(f"  {col_name:<35} {null_count:>10,}  ({pct:5.1f}%)")

display(dim_fin_geo.limit(10))

StatementMeta(, bff923fa-ee9b-41f5-9310-92a1660ca879, 5, Finished, Available, Finished, False)

Dim_FinancialGeography rows: 1,082

Column NULL counts:
-------------------------------------------------------
  GlobalFinancialGeographyId                   0  (  0.0%)
  Region                                       0  (  0.0%)
  SubRegion                                  401  ( 37.1%)
  CountryGroup                               731  ( 67.6%)
  Cluster                                    401  ( 37.1%)
  Country                                      0  (  0.0%)
  CountryCode                                  0  (  0.0%)


SynapseWidget(Synapse.DataFrame, d7959431-5d1f-4ce5-8b40-c49a9074d042)

In [4]:
# Cell 4: Data Quality Overview

print("=== Region Distribution ===")
display(
    dim_fin_geo.groupBy("Region")
    .agg(
        F.count("*").alias("Count"),
        F.round(F.count("*") / row_count * 100, 1).alias("Pct"),
    )
    .orderBy("Count", ascending=False)
)

print("\n=== SubRegion Distribution ===")
display(
    dim_fin_geo.groupBy("SubRegion")
    .agg(
        F.count("*").alias("Count"),
        F.round(F.count("*") / row_count * 100, 1).alias("Pct"),
    )
    .orderBy("Count", ascending=False)
)

print("\n=== Top 10 Countries ===")
display(
    dim_fin_geo.groupBy("Country")
    .agg(
        F.count("*").alias("Count"),
        F.round(F.count("*") / row_count * 100, 1).alias("Pct"),
    )
    .orderBy("Count", ascending=False)
    .limit(10)
)

StatementMeta(, bff923fa-ee9b-41f5-9310-92a1660ca879, 6, Finished, Available, Finished, False)

=== Region Distribution ===


SynapseWidget(Synapse.DataFrame, 4fffb0ff-1c35-4433-93d3-dd19a4c4e88f)


=== SubRegion Distribution ===


SynapseWidget(Synapse.DataFrame, 6deab0ef-0ba3-4056-9790-1fc135e2f7fe)


=== Top 10 Countries ===


SynapseWidget(Synapse.DataFrame, 55ff944c-ae2a-40a9-806c-c0bee1aa860f)

In [5]:
# Cell 5: Write to Gold Lakehouse

dim_fin_geo.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "Gold_SalesOps_Dim_FinancialGeography"
)

final_count = spark.read.table("Gold_SalesOps_Dim_FinancialGeography").count()
print(f"Gold_SalesOps_Dim_FinancialGeography written: {final_count:,} rows")
print("=== Notebook 02 complete ===")

StatementMeta(, bff923fa-ee9b-41f5-9310-92a1660ca879, 7, Finished, Available, Finished, False)

Gold_SalesOps_Dim_FinancialGeography written: 1,082 rows
=== Notebook 02 complete ===
